# Team Libra — 백테스트 데이터 수집

**기간**: 2019-01-01 ~ 2024-12-31  
**학습/검증 분리**: 2019~2022 train / 2023~2024 test  
**소스**: FinanceDataReader (KRX)

## 출력 파일
- `data/etf_2019_2024.parquet` — 5종 ETF
- `data/kospi200_index_2019_2024.parquet` — KOSPI 200 지수
- `data/kospi200_constituents_2019_2024.parquet` — KOSPI 200 구성종목 200개
- `data/stocks_2019_2024.parquet` — 개별 주식 5종

## 주의사항
- **Survivor bias**: `fdr.StockListing('KOSPI200')`은 *현재* 시점 구성종목임. 2019년 당시 편입돼있다가 빠진 종목은 포함 안 됨. 캡스톤 수준에서는 OK, 발표 limitation으로 명시.
- **거래비용**: 데이터 수집 단계에서는 raw price만 받음. 거래비용은 백테스트 *실행* 단계에서 별도 적용.

In [ ]:
# 필요 패키지 설치 (이미 설치돼 있으면 skip)
# !pip install finance-datareader pandas pyarrow tqdm matplotlib

In [ ]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np
from pathlib import Path
import time
from tqdm.notebook import tqdm

# 설정
START_DATE = '2019-01-01'
END_DATE = '2024-12-31'
TRAIN_END = '2022-12-31'  # 학습 구간 끝
SLEEP_SEC = 0.2  # API rate limit 방지

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f'기간: {START_DATE} ~ {END_DATE}')
print(f'학습: ~ {TRAIN_END}, 검증: {TRAIN_END} 이후')
print(f'저장 경로: {DATA_DIR.absolute()}')

## 유틸 함수

In [ ]:
def add_split_column(df, train_end=TRAIN_END):
    """학습/검증 split 컬럼 추가"""
    df = df.copy()
    df['split'] = np.where(df.index <= pd.Timestamp(train_end), 'train', 'test')
    return df

def fetch_one(ticker, name=None):
    """단일 티커 가격 수집. 실패 시 None."""
    try:
        df = fdr.DataReader(ticker, START_DATE, END_DATE)
        if df.empty:
            print(f'⚠ {ticker} ({name}): empty data')
            return None
        df = df.copy()
        df['ticker'] = ticker
        df['name'] = name or ticker
        return df
    except Exception as e:
        print(f'❌ {ticker} ({name}): {e}')
        return None

def fetch_many(tickers_dict, desc='Fetching'):
    """여러 티커 일괄 수집. tickers_dict = {ticker: name}"""
    frames = []
    failed = []
    for ticker, name in tqdm(tickers_dict.items(), desc=desc):
        df = fetch_one(ticker, name)
        if df is not None:
            frames.append(df)
        else:
            failed.append(ticker)
        time.sleep(SLEEP_SEC)
    if not frames:
        raise RuntimeError(f'{desc}: 모든 티커 실패')
    combined = pd.concat(frames)
    combined = add_split_column(combined)
    return combined, failed

## 1. 5종 ETF 수집

| 티커 | 이름 | 자산군 |
|---|---|---|
| 069500 | KODEX 200 | 한국 주식 |
| 360750 | TIGER 미국S&P500 | 미국 주식 |
| 133690 | TIGER 미국나스닥100 | 미국 성장주 |
| 114260 | KODEX 국고채3년 | 한국 채권 |
| 132030 | KODEX 골드선물 | 원자재(금) |

In [ ]:
ETF_TICKERS = {
    '069500': 'KODEX 200',
    '360750': 'TIGER 미국S&P500',
    '133690': 'TIGER 미국나스닥100',
    '114260': 'KODEX 국고채3년',
    '132030': 'KODEX 골드선물',
}

etf_df, etf_failed = fetch_many(ETF_TICKERS, desc='ETF')
etf_df.to_parquet(DATA_DIR / 'etf_2019_2024.parquet')

print(f'\n✓ ETF 저장: {len(etf_df):,} rows / {etf_df["ticker"].nunique()} tickers')
if etf_failed:
    print(f'⚠ 실패: {etf_failed}')
etf_df.head()

## 2. KOSPI 200 지수 (벤치마크)

In [ ]:
kospi200_index = fdr.DataReader('KS200', START_DATE, END_DATE)
kospi200_index['ticker'] = 'KS200'
kospi200_index['name'] = 'KOSPI 200'
kospi200_index = add_split_column(kospi200_index)
kospi200_index.to_parquet(DATA_DIR / 'kospi200_index_2019_2024.parquet')

print(f'✓ KOSPI 200 지수: {len(kospi200_index):,} rows')
print(f'  기간: {kospi200_index.index.min().date()} ~ {kospi200_index.index.max().date()}')
kospi200_index.head()

## 3. KOSPI 200 구성종목 200개

⚠ **소요시간**: 약 5-10분 (200종목 × 0.2초 sleep + API 응답시간)

In [ ]:
# 구성종목 리스트
kospi200_list = fdr.StockListing('KOSPI200')
print(f'현재 시점 KOSPI 200 구성종목: {len(kospi200_list)}개')
kospi200_list.head()

In [ ]:
# 200종목 일괄 수집
constituent_tickers = dict(zip(kospi200_list['Code'], kospi200_list['Name']))

constituents_df, kospi200_failed = fetch_many(constituent_tickers, desc='KOSPI200 구성종목')
constituents_df.to_parquet(DATA_DIR / 'kospi200_constituents_2019_2024.parquet')

print(f'\n✓ 구성종목 저장: {len(constituents_df):,} rows / {constituents_df["ticker"].nunique()} tickers')
if kospi200_failed:
    print(f'⚠ 실패한 티커 ({len(kospi200_failed)}개): {kospi200_failed[:10]}...')

## 4. 개별 주식 5종 (실제 보유)

| 티커 | 종목명 |
|---|---|
| 005930 | 삼성전자 |
| 293490 | 카카오게임즈 |
| 003490 | 대한항공 |
| 091160 | KODEX 반도체 |
| 192410 | 제룡전기 |

In [ ]:
STOCK_TICKERS = {
    '005930': '삼성전자',
    '293490': '카카오게임즈',
    '003490': '대한항공',
    '091160': 'KODEX 반도체',
    '192410': '제룡전기',
}

stocks_df, stocks_failed = fetch_many(STOCK_TICKERS, desc='개별주식')
stocks_df.to_parquet(DATA_DIR / 'stocks_2019_2024.parquet')

print(f'\n✓ 개별주식 저장: {len(stocks_df):,} rows / {stocks_df["ticker"].nunique()} tickers')
if stocks_failed:
    print(f'⚠ 실패: {stocks_failed}')
stocks_df.head()

## 5. 검증 — 저장된 데이터 점검

In [ ]:
print('=' * 70)
print('백테스트 데이터 수집 결과')
print('=' * 70)

for fpath in sorted(DATA_DIR.glob('*.parquet')):
    df = pd.read_parquet(fpath)
    print(f'\n📂 {fpath.name}')
    print(f'   행 수: {len(df):,}')
    print(f'   티커 수: {df["ticker"].nunique()}')
    print(f'   기간: {df.index.min().date()} ~ {df.index.max().date()}')
    if 'split' in df.columns:
        split_counts = df["split"].value_counts().to_dict()
        print(f'   학습/검증: {split_counts}')
    print(f'   컬럼: {list(df.columns)}')
    null_count = df.isnull().sum().sum()
    if null_count > 0:
        print(f'   ⚠ 결측치: {null_count}개')

print('\n' + '=' * 70)
print('완료 ✓')
print('=' * 70)

## 6. (선택) 시각화로 sanity check

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows. Mac이면 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ETF 5종 정규화 가격
etf = pd.read_parquet(DATA_DIR / 'etf_2019_2024.parquet')
for ticker in etf['ticker'].unique():
    sub = etf[etf['ticker'] == ticker]
    normalized = sub['Close'] / sub['Close'].iloc[0] * 100
    axes[0, 0].plot(sub.index, normalized, label=sub['name'].iloc[0], linewidth=1)
axes[0, 0].set_title('ETF 5종 — 정규화 가격 (시작=100)')
axes[0, 0].legend(fontsize=8)
axes[0, 0].axvline(pd.Timestamp(TRAIN_END), color='red', linestyle='--', alpha=0.5)
axes[0, 0].grid(alpha=0.3)

# KOSPI 200 지수
ks = pd.read_parquet(DATA_DIR / 'kospi200_index_2019_2024.parquet')
axes[0, 1].plot(ks.index, ks['Close'], color='navy')
axes[0, 1].axvline(pd.Timestamp(TRAIN_END), color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_title('KOSPI 200 지수')
axes[0, 1].grid(alpha=0.3)

# 개별주식 5종
st = pd.read_parquet(DATA_DIR / 'stocks_2019_2024.parquet')
for ticker in st['ticker'].unique():
    sub = st[st['ticker'] == ticker]
    normalized = sub['Close'] / sub['Close'].iloc[0] * 100
    axes[1, 0].plot(sub.index, normalized, label=sub['name'].iloc[0], linewidth=1)
axes[1, 0].set_title('개별주식 5종 — 정규화 가격 (시작=100)')
axes[1, 0].legend(fontsize=8)
axes[1, 0].axvline(pd.Timestamp(TRAIN_END), color='red', linestyle='--', alpha=0.5)
axes[1, 0].grid(alpha=0.3)

# KOSPI 200 구성종목 종가 분포
cs = pd.read_parquet(DATA_DIR / 'kospi200_constituents_2019_2024.parquet')
last_prices = cs.groupby('ticker').last()['Close']
axes[1, 1].hist(np.log10(last_prices), bins=30, color='steelblue', edgecolor='black')
axes[1, 1].set_title(f'KOSPI 200 구성종목 종가 분포 (log10, n={len(last_prices)})')
axes[1, 1].set_xlabel('log10(종가)')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()